# Data transformation 💄

## What you will learn in this course 🧐🧐

Raw business data is rarely ready to use. It comes in one row per event. Managers want one row per customer, per product, or per day. To go from one to the other, you need three core skills: **grouping** (to put rows together), **aggregation** (to summarize each group), and **handling missing values** (to deal with gaps in the data). These three skills are the base of every data pipeline. Without them, no clean report and no reliable model.

In this lecture, you work on real streaming platform data. Two files: one for viewing events, one for user profiles. By the end, you will be able to:

- Apply `groupby()` and `agg()` to summarize events by user, by content, or by event type.
- Build multi-level groupings to compare metrics across two or more dimensions.
- Detect missing values with `isnull()` and decide how to handle them.
- Choose between filling, dropping, or interpolating missing data based on business context.
- Combine events with user profiles using `merge()` to enrich your analysis.

## Business context

You work as a AI analyst at a streaming platform. The product team needs answers every Monday morning. 

> Which shows are people finishing? 

> Which users are most active? 

> Where is the experience breaking?

Your raw data sits in two CSV files.

The first file, `viewing_events.csv`, has **1,000 rows**. Each row is one event from one user. Events include `play`, `pause`, `stop`, `buffer`, and `error`. For each event, the platform records how long the user watched, how long the content lasts, and a user rating.

The second file, `user_profiles.csv`, has **50 rows**. One row per user. It tells you the subscription type (`basic`, `standard`, or `premium`), the country, and the monthly revenue.

> Your job: turn 1,000 raw events into clear answers about users, content, and quality.

Let's load the data.

In [1]:
import pandas as pd
import numpy as np

# Load both files
events = pd.read_csv('src/viewing_events.csv')
profiles = pd.read_csv('src/user_profiles.csv')

print(f"Events: {events.shape[0]} rows, {events.shape[1]} columns")
print(f"Profiles: {profiles.shape[0]} rows, {profiles.shape[1]} columns")
print("\nEvent columns:", list(events.columns))

Events: 1000 rows, 11 columns
Profiles: 50 rows, 5 columns

Event columns: ['user_id', 'timestamp', 'event_type', 'duration_seconds', 'watched_duration', 'total_duration', 'content_id', 'episode_number', 'watched_seconds', 'total_seconds', 'user_rating']


<Note type="tip">

Always print the shape and the column names right after loading data. This 2-second check catches surprises early. Wrong file, wrong format, or unexpected columns become visible before you build any logic on top.

</Note>

## Grouping and aggregation

<img src="https://ai-fullstack-assets.s3.eu-west-3.amazonaws.com/M02-EDA/AIFS-M02-D02-Group_By.png" />

**Grouping** means: *put rows together based on a shared value*. For example, group all rows that have the same `user_id`. 

**Aggregation** means: *for each group, compute one number*. For example, count how many rows are in the group, or take the average of one column.

These two steps almost always go together. You group, then you aggregate. The pandas method that does this is `groupby()`.

### Simple grouping

Let's start with a basic question. *Which users generate the most events?* You group the 1,000 rows by `user_id`, then count how many rows fall into each group.

In [2]:
# Count events per user
events_per_user = events.groupby('user_id').size()

# Sort to see the most active users first
events_per_user_sorted = events_per_user.sort_values(ascending=False)

print("Top 5 most active users:")
print(events_per_user_sorted.head(5))

Top 5 most active users:
user_id
user_050    28
user_037    28
user_020    27
user_036    27
user_022    26
dtype: int64


Here is what each line does.

`events.groupby('user_id')` does **not** compute anything yet. It just creates a *group object* that knows how to split the rows by user. Think of it as a recipe, not a result.

`.size()` is the aggregation. It counts how many rows are in each group. The output is a **Series** (a single column with labels). The label is the `user_id`. The value is the row count.

`.sort_values(ascending=False)` reorders the result with the highest count on top. `user_037` and `user_050` each have **28 events**. They are the most active users in the dataset.

<Note type="important">

`groupby()` alone does not return a DataFrame you can read. You must always follow it with an aggregation: `.size()`, `.sum()`, `.mean()`, `.count()`, or `.agg()`. Without the aggregation step, you only have a recipe.

</Note>

### Multiple statistics in one call

Often you want more than one number per group. The product team asks: *for each event type, what is the average watched time, and what is the average rating?* You can answer both questions in a single call using `.agg()`.

In [3]:
# Group by event type and compute several stats at once
stats_by_event = events.groupby('event_type').agg({
    'duration_seconds': 'mean',
    'watched_duration': 'mean',
    'user_rating': 'mean'
})

stats_by_event = stats_by_event.round(2)

print("Average values per event type:")
print(stats_by_event)

Average values per event type:
            duration_seconds  watched_duration  user_rating
event_type                                                 
buffer                915.53            428.07         2.91
error                 907.48            405.49         2.95
pause                 934.47            421.07         2.99
play                 3769.08           2504.47         3.02
stop                  968.24            432.32         3.20


`.agg()` accepts a dictionary. The **key** is the column to aggregate. The **value** is the function to apply. Here, you ask for the mean of three numeric columns.

`.round(2)` rounds every number to 2 decimal places. This makes the table easier to read. Always round before you show numbers to a non-technical audience.

Now read the result with business eyes:

- `play` events have an average watched time of **2,504 seconds**, much higher than other event types. This makes sense: a `play` event is a real watching session.
- `error` events have the lowest average rating (**2.95**) and `stop` events the highest (**3.20**). Errors push satisfaction down.
- `buffer` events have a low rating too (**2.91**). Quality issues hurt the user experience.

> One groupby call, three business insights. This is the power of aggregation.

### Multi-level grouping

Sometimes one column is not enough. You may want to compare values across **two dimensions** at the same time. For example, *for each user, how many events of each type do they generate?* You group by two columns at once.

In [4]:
# Group by user AND event type at the same time
user_event_stats = events.groupby(['user_id', 'event_type']).agg(
    event_count=('duration_seconds', 'count'),
    avg_watched=('watched_duration', 'mean')
)

user_event_stats = user_event_stats.round(2)

print("Events per user and per event type (first 10 rows):")
print(user_event_stats.head(10))

Events per user and per event type (first 10 rows):
                     event_count  avg_watched
user_id  event_type                          
user_001 buffer                2       502.50
         error                 3       485.33
         pause                 6       373.33
         play                  4      3323.50
         stop                  4       650.75
user_002 buffer                1       337.00
         error                 2       248.00
         pause                 3       505.33
         play                  3      1935.00
         stop                  2       276.50


Here is what changed compared to the simple groupby.

`groupby(['user_id', 'event_type'])` takes a **list of two columns**. The result has one row per unique combination of user and event type. `user_001` will appear up to 5 times in the output (once per event type they triggered).

`.agg(event_count=('duration_seconds', 'count'), avg_watched=('watched_duration', 'mean'))` uses **named aggregation**. Each new column has a clear name. `event_count` counts the rows. `avg_watched` is the mean of `watched_duration`. This syntax is cleaner than the dictionary form. Use it when output names matter.

Looking at `user_001`, you can see they triggered **4 play events** with an average of **3,323 seconds watched**, but also **6 pause events** and **3 errors**. That tells a fuller story than a single number per user.

<Note type="tip">

Multi-level grouping creates a *MultiIndex* on rows. To turn the result back into a regular table, add `.reset_index()` at the end. The two grouping columns become regular columns again.

</Note>

### Ranking content by total watch time

A frequent business question is: *which content drives the most viewing minutes?* You group by `content_id`, sum up the watched durations, and sort the result.

In [5]:
# Compute three metrics per content piece
content_stats = events.groupby('content_id').agg(
    total_views=('user_id', 'count'),
    avg_rating=('user_rating', 'mean'),
    total_watched_min=('watched_duration', 'sum')
)

# Convert seconds to minutes for easier reading
total_minutes = content_stats['total_watched_min'] / 60
content_stats['total_watched_min'] = total_minutes.round(1)

content_stats['avg_rating'] = content_stats['avg_rating'].round(2)

# Sort by watch time, top 5 first
content_stats_sorted = content_stats.sort_values('total_watched_min', ascending=False)

print("Top 5 contents by total watch time (minutes):")
print(content_stats_sorted.head(5))

Top 5 contents by total watch time (minutes):
             total_views  avg_rating  total_watched_min
content_id                                             
content_014           16        2.69              384.4
content_061           23        3.31              336.1
content_047            9        2.24              300.0
content_029           12        2.58              299.8
content_084           11        2.94              298.2


This is a complete business report in 5 lines of pandas.

`total_views=('user_id', 'count')` counts how many events the content received. It uses `user_id` only because counting any non-null column gives the same answer. `avg_rating=('user_rating', 'mean')` computes the average rating. `total_watched_min=('watched_duration', 'sum')` adds up all the seconds users actually watched.

Then you divide by 60 to convert seconds to minutes. You store the division result in a separate variable `total_minutes` first, then assign it back to the column. Each line does one thing.

Reading the output:

- `content_014` is the leader with **384 minutes** of total watch time across **16 views**, but the rating is only **2.69**. *Popular but not loved.*
- `content_061` got **23 views** (more than `content_014`) but less total watch time. *Many users started, fewer finished.*
- `content_047` has the lowest rating among the top 5 (**2.24**). Investigate why.

## Missing values

Real data has gaps. A sensor fails. A user skips a rating. A network drops a packet. These gaps show up as `NaN` (Not a Number) in pandas. *NaN* is a special floating-point value that means "missing".

<Note type ="tip">
Missing values are not a small issue. In **finance**, a missing transaction type can hide whether money came in or went out. In **healthcare**, a missing measurement can change a diagnosis. **You must never ignore missing values.** You must always check, understand why they are missing, and decide how to handle them.

<img src ="https://data-analytics-fullstack-assets.s3.eu-west-3.amazonaws.com/M05-Exploratory_Data_Analysis/M3_D2_Missing_Values.png"/>

This is why data cleaning is never just a technical exercise. You cannot modify or fill in missing values blindly. Every decision must be guided by the **context and the domain knowledge** that explain what the data truly represents.

If you are curious about missing value imputation, here are some useful resources to explore:
- [A Gentle Introduction to Handling Missing Data](https://machinelearningmastery.com/handle-missing-data-python/)
- [Dealing with Missing Data](https://www.kaggle.com/code/residentmario/dealing-with-missing-data/notebook)
- [Imputation de données manquantes](https://www.math.univ-toulouse.fr/~besse/Wikistat/pdf/st-m-app-idm.pdf)

</Note>

Let's first check the original file.

In [6]:
# Step 1: count missing values per column
missing_counts = events.isnull().sum()

print("Missing values per column:")
print(missing_counts)

Missing values per column:
user_id             0
timestamp           0
event_type          0
duration_seconds    0
watched_duration    0
total_duration      0
content_id          0
episode_number      0
watched_seconds     0
total_seconds       0
user_rating         0
dtype: int64


`isnull()` returns a DataFrame of the same shape as `events`, but every cell is either `True` (missing) or `False` (filled). `.sum()` then adds up the `True` values per column. The result is a count of missing values for each column.

Good news: the original file is clean. **Zero missing values everywhere.** But this is rare in real production data. So let's create a more realistic version of the dataset with gaps, then learn how to handle them.

In [7]:
# Create a copy so we keep the clean version untouched
events_with_gaps = events.copy()

# Set a seed so the random gaps are reproducible
np.random.seed(42)

# Drop 80 ratings (users skipped rating)
missing_rating_index = np.random.choice(events_with_gaps.index, 80, replace=False)

# Add NaN to the user_rating column for the selected indices
events_with_gaps.loc[missing_rating_index, 'user_rating'] = np.nan

# Drop 40 watched_duration values (sensor or log dropouts)
missing_watch_index = np.random.choice(events_with_gaps.index, 40, replace=False)

# Add NaN to the watched_duration column for the selected indices
events_with_gaps.loc[missing_watch_index, 'watched_duration'] = np.nan

print("Missing values after introducing realistic gaps:")
print(events_with_gaps.isnull().sum())

Missing values after introducing realistic gaps:
user_id              0
timestamp            0
event_type           0
duration_seconds     0
watched_duration    40
total_duration       0
content_id           0
episode_number       0
watched_seconds      0
total_seconds        0
user_rating         80
dtype: int64


Now you have **80 missing ratings** and **40 missing watched durations**. This looks like real data.

Now the question is: *what do you do with these gaps?* You have three main options. The right choice depends on the column.

### Option 1: fill missing values

**Use case:** the rating is missing because the user skipped the rating step. The event itself is still valid. You can fill the gap with a reasonable value, for example the **mean rating** of the dataset.

In [8]:
# Compute the mean rating across non-missing rows
mean_rating = events_with_gaps['user_rating'].mean()
print(f"Mean rating: {mean_rating:.2f}")

# Fill the gaps with that mean
filled = events_with_gaps.copy()
filled['user_rating'] = filled['user_rating'].fillna(mean_rating)

print("\nMissing values after filling ratings:")
print(filled.isnull().sum())

Mean rating: 3.01

Missing values after filling ratings:
user_id              0
timestamp            0
event_type           0
duration_seconds     0
watched_duration    40
total_duration       0
content_id           0
episode_number       0
watched_seconds      0
total_seconds        0
user_rating          0
dtype: int64


`.mean()` automatically skips `NaN` values, so the average uses only the 920 valid ratings. The mean rating is **3.01**.

`.fillna(value)` replaces every `NaN` in the column with the value you pass. After this step, the `user_rating` column has zero missing values. The `watched_duration` column still has 40 gaps because we did not touch it.

<Note type="important">

Filling with the mean is simple but **biased**. Every imputed row now sits exactly on the average, which artificially shrinks the variance. For a basic dashboard this is fine. For statistical modeling, prefer methods like **median fill**, **forward fill** (`ffill`), or **interpolation**.

</Note>

### Option 2: drop the missing rows

`watched_duration` is the core metric. If it is missing, the event is useless for engagement analysis. There is no safe way to invent a watched time. So you drop those rows.

In [9]:
# Drop only rows where watched_duration is missing
before_count = len(filled)
cleaned = filled.dropna(subset=['watched_duration'])
after_count = len(cleaned)

rows_dropped = before_count - after_count
print(f"Before: {before_count} rows")
print(f"After:  {after_count} rows")
print(f"Dropped: {rows_dropped} rows")

print("\nMissing values in the cleaned table:")
print(cleaned.isnull().sum())

Before: 1000 rows
After:  960 rows
Dropped: 40 rows

Missing values in the cleaned table:
user_id             0
timestamp           0
event_type          0
duration_seconds    0
watched_duration    0
total_duration      0
content_id          0
episode_number      0
watched_seconds     0
total_seconds       0
user_rating         0
dtype: int64


`.dropna(subset=['watched_duration'])` removes a row **only if** `watched_duration` is missing. Other columns can still have `NaN`, but here we already filled `user_rating` so the cleaned table has zero gaps.

You went from **1,000 rows to 960 rows**. You lost 4% of the data. Acceptable for engagement analysis. Not acceptable if those 40 missing rows are concentrated in one user or one country (you would introduce bias).

<Note type="tip">

Before dropping rows, always check whether the missingness is **random** or **patterned**. Run `events_with_gaps[events_with_gaps['watched_duration'].isnull()].groupby('event_type').size()` to see if the gaps cluster in one event type. If they do, dropping will skew your results.

</Note>

### Option 3: forward fill or interpolate

**Use case:** time-ordered data. If the speed sensor of a car drops a reading, the value 100 milliseconds before is usually a good guess. Pandas offers `ffill()` (carry the last valid value forward) and `interpolate()` (estimate based on neighbors).

Here is a small example using a small slice of data, sorted by time.

In [10]:
# Take a small ordered slice for one user
user_one = events_with_gaps[events_with_gaps['user_id'] == 'user_001']
user_one_sorted = user_one.sort_values('timestamp')

# Apply forward fill on watched_duration
user_one_filled = user_one_sorted.copy()
user_one_filled['watched_duration'] = user_one_filled['watched_duration'].ffill()

# Compare a few rows
compare_cols = ['timestamp', 'event_type', 'watched_duration']
print("Original (with possible gaps):")
print(user_one_sorted[compare_cols].head(8))
print("\nAfter forward fill:")
print(user_one_filled[compare_cols].head(8))

Original (with possible gaps):
               timestamp event_type  watched_duration
383  2024-01-17 12:55:00       stop            1341.0
543  2024-01-27 21:40:00       play            3737.0
401  2024-01-30 17:22:00       play            5090.0
204  2024-01-31 18:26:00       stop              87.0
395  2024-02-01 01:30:00       play            2552.0
669  2024-02-06 19:01:00       play            1915.0
269  2024-02-07 18:43:00       stop             556.0
399  2024-02-07 18:50:00      pause             446.0

After forward fill:
               timestamp event_type  watched_duration
383  2024-01-17 12:55:00       stop            1341.0
543  2024-01-27 21:40:00       play            3737.0
401  2024-01-30 17:22:00       play            5090.0
204  2024-01-31 18:26:00       stop              87.0
395  2024-02-01 01:30:00       play            2552.0
669  2024-02-06 19:01:00       play            1915.0
269  2024-02-07 18:43:00       stop             556.0
399  2024-02-07 18:50:00      

`ffill()` walks down the column and copies the **last valid value** into the next `NaN` cell. If there are several `NaN` in a row, they all get the same value (the last one before the gap).

`interpolate()` is smarter for numeric data. It draws a straight line between the value before and the value after, and fills the gap with a point on that line. Use it for sensor readings, prices, or any continuous signal.

<Note type="important">

`ffill()` and `interpolate()` only make sense on **sorted** data. If your rows are not in order, the value you carry forward is meaningless. Always `.sort_values()` before filling time-ordered data.

</Note>

## Combining datasets with merge

Most business questions need data from more than one table. *Do premium users rate higher than basic users?* The rating lives in `events`. The subscription type lives in `profiles`. To answer the question, you must combine the two tables.

<img src ="https://ai-fullstack-assets.s3.eu-west-3.amazonaws.com/M02-EDA/AIFS-M02-D02-Merge_joins.png"/>

The pandas method for this is `merge()`. It is the equivalent of a SQL `JOIN`.

In [11]:
# Combine events with user profiles using user_id as the link
merged = events.merge(profiles, on='user_id', how='left')

preview_cols = ['user_id', 'event_type', 'subscription_type', 'country', 'monthly_revenue']
print("First 5 merged rows:")
print(merged[preview_cols].head(5))

print(f"\nMerged shape: {merged.shape}")

First 5 merged rows:
    user_id event_type subscription_type country  monthly_revenue
0  user_041       play           premium      FR            17.99
1  user_035      error          standard      JP             8.99
2  user_002      pause          standard      CA             8.99
3  user_001      pause           premium      DE            13.99
4  user_022       play          standard      DE            13.99

Merged shape: (1000, 15)


`events.merge(profiles, on='user_id', how='left')` connects the two tables.

`on='user_id'` tells pandas to match rows where the `user_id` is the same in both tables. This column must exist in both. It is called the **join key**.

`how='left'` means: *keep every row from the left table (`events`), and add matching info from the right table (`profiles`)*. If a `user_id` from `events` is missing in `profiles`, the new columns get `NaN`. With `how='left'` you never lose events.

There are four common join types:

- **`left`**: keep all left rows, attach matches from the right. Most common when enriching event data.
- **`right`**: keep all right rows, attach matches from the left. Mirror of left.
- **`inner`**: keep only rows that exist in both tables. Drops mismatches.
- **`outer`**: keep all rows from both tables. Fill missing matches with `NaN`.

Now use the merged table to answer the original business question.

In [12]:
# Group by subscription type and compute three business metrics
by_subscription = merged.groupby('subscription_type').agg(
    total_events=('user_id', 'count'),
    avg_rating=('user_rating', 'mean'),
    avg_revenue=('monthly_revenue', 'mean')
)

by_subscription = by_subscription.round(2)

print("Engagement and revenue by subscription type:")
print(by_subscription)

Engagement and revenue by subscription type:
                   total_events  avg_rating  avg_revenue
subscription_type                                       
basic                       268        2.85        14.17
premium                     390        3.09        13.60
standard                    342        3.06        13.09


Now you have a clear answer for the product team.

- **Premium** users generate the most events (**390**) and rate the highest (**3.09**). They are the most engaged segment.
- **Standard** users come close on rating (**3.06**) but generate fewer events (**342**).
- **Basic** users are the smallest cohort (**268 events**) and rate the lowest (**2.85**).

> Premium users are not just paying more, they are also enjoying the product more. The product team can use this to justify the premium tier strategy.

<Note type="important">

After every merge, always check the row count. If the shape grew unexpectedly, you probably have **duplicates** in one of the tables that caused a one-to-many or many-to-many explosion. Run `profiles['user_id'].duplicated().sum()` before merging to catch this early.

</Note>

## Resources 📚📚

- [pandas.pydata.org](https://pandas.pydata.org/docs/)
- [A Gentle Introduction to Handling Missing Data](https://machinelearningmastery.com/handle-missing-data-python/)
- [Missing Data and Multiple Imputation](https://www.publichealth.columbia.edu/research/population-health-methods/missing-data-and-multiple-imputation). 